# Лабораторная работа №9
## Обучение без учителя: PCA, t-SNE и кластеризация

**Задание:** датасет **D1** (подмножество признаков, >2, без целевой переменной) → **PCA** и **t-SNE** в 2D (**D2**, **D3**) → визуализация и сравнение «разделимости» → кластеризация **не менее трёх** методов на **D1, D2, D3** → метрики качества → выводы.


## 1. Датасет D1

Используем **Wine** (`sklearn.datasets.load_wine`): **13 числовых признаков**, целевой признак (сорт вина) **не входит** в D1 — он нужен только для **иллюстрации** на рисунках (истинные классы) и не используется при обучении PCA/t-SNE/кластеризации.

Количество признаков 13 > 2, условие выполнено. Данные **стандартизируем** (`StandardScaler`), так как признаки в разных масштабах.


In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

RANDOM_STATE = 42
N_CLUSTERS = 3  # в Wine три класса винограда — задаём k для методов, требующих число кластеров

wine = load_wine()
X_raw = wine.data
feature_names = wine.feature_names
y_true = wine.target

D1 = pd.DataFrame(X_raw, columns=feature_names)
print("D1: размер", D1.shape)
display(D1.describe().T.round(3))


In [ ]:
scaler = StandardScaler()
D1_z = scaler.fit_transform(D1.values)

# Именованно храним как DataFrame для наглядности (исходный масштаб — только D1_z)
D1_df = pd.DataFrame(D1_z, columns=feature_names)
print("D1 после StandardScaler: среднее ~0, дисперсия ~1 по столбцам")


## 2. Снижение размерности: D2 (PCA) и D3 (t-SNE)

Обе проекции в **две** главные координаты.


In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
D2 = pca.fit_transform(D1_z)
print("Доля объяснённой дисперсии (PC1+PC2):", round(pca.explained_variance_ratio_.sum(), 4))

tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=RANDOM_STATE,
    max_iter=1000,
)
D3 = tsne.fit_transform(D1_z)

D2_df = pd.DataFrame(D2, columns=["PC1", "PC2"])
D3_df = pd.DataFrame(D3, columns=["z1", "z2"])
print("D2:", D2.shape, "D3:", D3.shape)


## 3. Визуализация D2 и D3

**Цветом** показаны **истинные классы** Wine (только для визуальной оценки «разделимости» кластеров).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
scatter_kw = dict(c=y_true, cmap="tab10", alpha=0.85, edgecolors="k", linewidths=0.3)

axes[0].scatter(D2[:, 0], D2[:, 1], **scatter_kw)
axes[0].set_title("D2: первые две главные компоненты (PCA)")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

axes[1].scatter(D3[:, 0], D3[:, 1], **scatter_kw)
axes[1].set_title("D3: t-SNE в 2D")
axes[1].set_xlabel("z1")
axes[1].set_ylabel("z2")

plt.suptitle("Проекции датасета D1 (цвет = класс вина, только визуализация)")
plt.tight_layout()
plt.show()


### Ответ на вопрос задания

Обычно на **Wine** классы **визуально сильнее разделены в проекции t-SNE (D3)**, чем в первых двух главных компонентах PCA (D2): t-SNE сохраняет локальную структуру и лучше разводит группы в 2D. PCA — линейная проекция с максимальной дисперсией; для этого датасета перекрытия классов на D2 часто заметнее.

Сравнение можно уточнить численно: например, оценить **силуэт** эталонной кластеризации (например, KMeans) отдельно в D2 и D3 — у кого выше silhouette при том же k, тот обычно «явнее» (при фиксированном k). Это сделано в следующем разделе.


## 4. Кластеризация: три метода

**KMeans**, **агломеративная кластеризация (Ward)**, **Gaussian Mixture (EM)**. Число кластеров `k=3` согласовано с естественной структурой Wine (три класса), как типичный выбор для сравнения метрик.


In [ ]:
def clustering_models(k: int, random_state: int = 42):
    return {
        "KMeans": KMeans(n_clusters=k, random_state=random_state, n_init="auto"),
        "Agglomerative (Ward)": AgglomerativeClustering(n_clusters=k, linkage="ward"),
        "GaussianMixture": GaussianMixture(n_components=k, random_state=random_state),
    }


def fit_labels(model, X):
    return model.fit_predict(X)


def score_clustering(X, labels):
    if len(np.unique(labels)) < 2:
        return np.nan, np.nan, np.nan
    sil = silhouette_score(X, labels)
    db = davies_bouldin_score(X, labels)
    ch = calinski_harabasz_score(X, labels)
    return sil, db, ch


datasets = {
    "D1 (13 признаков, стандартизировано)": D1_z,
    "D2 (PCA-2D)": D2,
    "D3 (t-SNE-2D)": D3,
}

models = clustering_models(N_CLUSTERS, RANDOM_STATE)
results = []

for dname, Xp in datasets.items():
    for mname, model in models.items():
        lab = fit_labels(model, Xp)
        sil, db, ch = score_clustering(Xp, lab)
        results.append(
            {
                "Датасет": dname,
                "Метод": mname,
                "Silhouette ↑": sil,
                "Davies-Bouldin ↓": db,
                "Calinski-Harabasz ↑": ch,
            }
        )

res_df = pd.DataFrame(results)
display(res_df.round(4))


### Визуальная «явность» D2 vs D3 при фиксированном KMeans

**Silhouette** выше — разделение объектов между кластерами **яснее** (для выбранного k).


In [ ]:
km_d2 = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init="auto")
km_d3 = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init="auto")

l2 = km_d2.fit_predict(D2)
l3 = km_d3.fit_predict(D3)
s2, _, _ = score_clustering(D2, l2)
s3, _, _ = score_clustering(D3, l3)

print("Silhouette KMeans(k=3): D2 (PCA) =", round(s2, 4), "| D3 (t-SNE) =", round(s3, 4))
print("Вывод сравнения 2D-проекций:", "t-SNE нагляднее по silhouette," if s3 > s2 else "PCA 2D даёт не ниже silhouette для KMeans,")


## 5. Сводка лучших методов по каждому датасету

По **Silhouette** (главный показатель из трёх для интерпретации разделимости); DB и CH — дополнительно.


In [ ]:
best_rows = []
for dname in res_df["Датасет"].unique():
    sub = res_df[res_df["Датасет"] == dname].copy()
    sub = sub.dropna(subset=["Silhouette ↑"])
    if sub.empty:
        continue
    i = sub["Silhouette ↑"].idxmax()
    best_rows.append(res_df.loc[i])

best_summary = pd.DataFrame(best_rows)
display(best_summary[["Датасет", "Метод", "Silhouette ↑", "Davies-Bouldin ↓", "Calinski-Harabasz ↑"]].round(4))


## 6. Выводы (шаблон для отчёта)

1. **D1 (исходное пространство признаков):** модели используют всю информацию; **KMeans / GMM / Ward** по-разному учитывают геометрию и плотности. Обычно **EM (GMM)** или **Ward** хорошо работают, если кластеры **выпуклые / иерархические**.
2. **D2 (PCA-2D):** линейная **проекция**; удобна для графика. Значения Silhouette/DB/CH **нельзя** напрямую сопоставлять с D1 из‑за другого пространства признаков и масштаба расстояний.
3. **D3 (t-SNE-2D):** часто **лучше видны группы** на scatter, но t-SNE **не сохраняет глобальные расстояния**; метрики на D3 полезны для сравнения методов **в этой проекции**, но слабо сопоставимы с D1 по абсолютному значению.
4. **Какой метод кластеризации «лучше»:** смотреть на **конкретные строки таблицы** — максимум **Silhouette**, минимум **Davies-Bouldin**, максимум **Calinski-Harabasz**. Расхождения между метриками возможны: тогда опирайтесь на Silhouette + предметный смысл Wine.

*Заполните абзац с фактическими числами из `res_df` после запуска ноутбука.*
